# 🎯 R1-Zero 强化学习实战：冷启动 SFT + GRPO 多奖励函数

> 本教程整理自 [jackrong《Jackrong-llm-finetuning-guide》](https://github.com/R6410418/Jackrong-llm-finetuning-guide) 仓库的
> `train_code/Llama-3.2-3B-R1-Zero-GRPO.ipynb`（仓库最有价值的 RL 教程），按 cook 风格重排。
> 复现 DeepSeek-R1-Zero 式推理训练：**先用 SFT 教格式，再用 GRPO 多奖励函数强化数学推理**。

| 项目 | 内容 |
|---|---|
| 🧠 基座模型 | `unsloth/Llama-3.2-3B-Instruct`（LoRA 16bit，r=64） |
| 🧪 训练范式 | 两阶段：冷启动 SFT（格式塑造）→ GRPO（群组相对策略优化） |
| 📚 训练数据 | `unsloth/OpenMathReasoning-mini`（冷启动）+ `open-r1/DAPO-Math-17k-Processed`（RL） |
| 🎁 奖励函数 | 4+1 个：结构塑造 / `\boxed{}` 答案校验 / 重复惩罚 / 采样监控 |

**目录**
1. R1-Zero 两阶段动机：为什么先 SFT 再 GRPO
2. 实验设计总览
3. 环境安装
4. 实验配置
5. 冷启动 SFT：格式契约 + 数据 + 训练
6. GRPO 数据预处理
7. 奖励函数四件套
8. GRPO 训练
9. 训练观察与推理自测
10. 保存与 GGUF 导出
11. 附录

## 1️⃣ R1-Zero 两阶段动机：为什么先 SFT 再 GRPO

DeepSeek-R1-Zero 证明了**纯 RL 就能涌现推理能力**，但直接用 3B 小模型做纯 GRPO 有一个现实问题：
模型连"在哪里放推理、在哪里放答案"都不会，奖励函数无从谈起。jackrong 的方案是两阶段：

```
 ┌────────────────────────────────────────────────────────────┐
 │  阶段 1：冷启动 SFT（OpenMathReasoning-mini）                │
 │  把每条样本包装成固定格式：                                  │
 │    <think>...分步推理...</think>                            │
 │    <answer>oxed{最终数值答案}</answer>                     │
 │  → 模型先学会"格式契约"，RL 开始时不是一团乱码               │
 ├────────────────────────────────────────────────────────────┤
 │  阶段 2：GRPO（DAPO-Math-17k）                              │
 │  对每组 N 条自生成回答，按奖励函数打分 → 组内相对优势 → 更新 │
 │  奖励矩阵：结构奖励(格式) + 答案校验(主信号) + 重复惩罚(弱)   │
 └────────────────────────────────────────────────────────────┘
```

> 💡 为什么"冷启动"能省算力：格式混乱是 RL 早期最大的探索浪费，几十步 SFT 就解决的格式问题，
> 纯 RL 可能要几百步。**先 SFT 后 RL 是 R1 系训练性价比最高的组合**（原 notebook 第 20 个 cell 的原话大意）。

## 2️⃣ 实验设计总览

| 决策 | 选择 | 原因 |
|---|---|---|
| 基座 | Llama-3.2-3B-Instruct | 3B 足够小，Kaggle 免费 GPU 可跑完整两阶段 |
| 量化 | `load_in_4bit=False`（LoRA 16bit） | 原注释：False for LoRA 16bit——3B 全 bf16 也就 6GB |
| fast_inference | True | Unsloth 挂 vLLM，GRPO 采样快一个量级 |
| LoRA | r=64 / α=2r | 原注释：*2 speeds up training |
| 序列长度 | 4096 | 推理轨迹可控；DAPO 数据按 90% 分位数截断 |
| 冷启动数据 | OpenMathReasoning-mini（cot split） | 只保留 `expected_answer` 是纯数字的行（奖励可校验） |
| RL 数据 | DAPO-Math-17k-Processed（en） | prompt 模板化好、带标准答案 |
| GRPO 采样 | num_generations=6 | 原注释：Decrease if out of memory |

## 3️⃣ 环境安装

> 原 notebook 运行在 **Kaggle**（`kaggle_secrets` 存 HF_TOKEN/WANDB_API_KEY），Colab 把第一行换成 `google.colab.userdata` 即可。
> 安装约 1-2 分钟，装完按提示重启运行时。

In [ ]:
# Kaggle 版（原 notebook）；Colab 换成：from google.colab import userdata; userdata.get('HF_TOKEN')
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

In [ ]:
%%capture
import importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None:
    !uv pip install -qqq torch torchvision
if importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth
!uv pip install -qqq vllm

## 4️⃣ 实验配置

本 cell 无 GPU 也能跑（配置单）。GRPO 的显存大头是**采样时的 vLLM 引擎**（`UNSLOTH_VLLM_STANDBY=1`
省 30%+ 显存，原 notebook 第一行）。

In [1]:
import os
os.environ['UNSLOTH_VLLM_STANDBY'] = "1"  # Unsloth Standby reduces VRAM by 30%+

from dataclasses import dataclass, field
import torch

@dataclass
class R1Config:
    # ---- 模型 ----
    model_name: str = "unsloth/Llama-3.2-3B-Instruct"
    max_seq_length: int = 4096          # Can increase for longer reasoning traces
    lora_rank: int = 64                 # Larger rank = smarter, but slower

    # ---- 冷启动 SFT ----
    cold_start_dataset: str = "unsloth/OpenMathReasoning-mini"
    cold_start_split: str = "cot"
    sft_epochs: int = 2
    sft_batch: int = 4
    sft_accum: int = 4                  # GA 模拟大 batch
    sft_lr: float = 2e-4                # 长训降 2e-5（原注释）

    # ---- GRPO ----
    grpo_dataset: str = "open-r1/DAPO-Math-17k-Processed"
    grpo_config: str = "en"
    temperature: float = 1.0
    grpo_lr: float = 5e-6               # RL 学习率远小于 SFT
    num_generations: int = 6            # Decrease if out of memory
    grpo_batch: int = 6
    grpo_accum: int = 6
    max_steps: int = 1500
    seed: int = 3407

cfg = R1Config()
torch.manual_seed(cfg.seed)
cfg

R1Config(model_name='unsloth/Llama-3.2-3B-Instruct', max_seq_length=4096, lora_rank=64, cold_start_dataset='unsloth/OpenMathReasoning-mini', cold_start_split='cot', sft_epochs=2, sft_batch=4, sft_accum=4, sft_lr=0.0002, grpo_dataset='open-r1/DAPO-Math-17k-Processed', grpo_config='en', temperature=1.0, grpo_lr=5e-06, num_generations=6, grpo_batch=6, grpo_accum=6, max_steps=1500, seed=3407)

## 5️⃣ 冷启动 SFT ①：格式契约

**目标格式**（模型输出必须严格匹配）：

```
<think>
... step-by-step reasoning ...
</think>
<answer>\boxed{最终数值答案}</answer>
```

关键点（原 notebook 注释）：
- system prompt 里明确写出格式契约，模板用 **Llama-3.x 原生 Jinja**（含反引号转义函数）；
- 从 `generated_solution` 里**尽量只抽推理部分**（`_extract_think_from_generated_solution`），
  `\boxed{}` 只放 `expected_answer`——防止上游答案污染 `<answer>` 格式；
- 推理太短（<5 字符）会补一句兜底话，避免模型学到"空 think 摆烂"。

In [2]:
# =========================================================
# ✅  Route B: 模型自主输出 <think>...</think><answer>...</answer>
#    system prompt + SFT 冷启动 + RL 格式奖励，三管齐下学格式
# =========================================================

THINK_OPEN  = "<think>"
THINK_CLOSE = "</think>"
ANSWER_OPEN  = "<answer>"
ANSWER_CLOSE = "</answer>"

system_prompt = f"""You are a helpful assistant.
You will be provided with a problem. Please follow this format exactly (do not add any extra text outside the tags):
{THINK_OPEN}
... step-by-step reasoning ...
{THINK_CLOSE}
{ANSWER_OPEN}\\boxed{{final numeric answer}}{ANSWER_CLOSE}
The <think> section can be short for easy problems, but must not be empty.
The final answer inside \\boxed{{}} should be a single number.
"""

def _escape_for_jinja_single_quote(s: str) -> str:
    return s.replace("\\", "\\\\").replace("'", "\\'").replace("\n", "\\n")

_SYSTEM_PROMPT_ESC = _escape_for_jinja_single_quote(system_prompt)

# ✅  Llama-3.x 原生 Chat Template（原 notebook 手写 Jinja，逐字保留）
chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
    "{{ '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n' + messages[0]['content'] + '<|eot_id|>' }}"
    "{% set loop_messages = messages[1:] %}"
    "{% else %}"
    "{{ '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n' + '" + _SYSTEM_PROMPT_ESC + "' + '<|eot_id|>' }}"
    "{% set loop_messages = messages %}"
    "{% endif %}"

    "{% for message in loop_messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ '<|start_header_id|>user<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
    "{% endif %}"
)

import re

def _extract_think_from_generated_solution(sol: str) -> str:
    """从 OpenMathReasoning-mini 的 generated_solution 里尽量只抽推理部分。
    \boxed{} 只放 expected_answer，防止上游答案污染 <answer> 格式。"""
    if sol is None:
        return ""
    s = str(sol)
    # 1) 优先取 <think>...</think> 内部
    m = re.search(r"<think>(.*?)</think>", s, flags=re.DOTALL)
    if m is not None:
        s = m.group(1)
    # 2) 去掉可能的 <answer>...</answer>
    s = re.sub(r"<answer>.*?</answer>", "", s, flags=re.DOTALL)
    # 3) 去掉常见 "最终答案" 片段
    s = re.sub(r"Final\s*Answer\s*:.*", "", s, flags=re.DOTALL | re.IGNORECASE)
    s = re.sub(r"####.*", "", s, flags=re.DOTALL)
    # 4) 出现 \boxed 直接截断（防止答案混进推理）
    s = s.split(r"\boxed", 1)[0]
    return s.strip()

def format_dataset(x):
    """一条 OpenMath 样本 → 冷启动训练消息（system + user + assistant 格式契约）。"""
    expected_answer = str(x["expected_answer"]).strip()
    problem = x["problem"]
    thoughts = _extract_think_from_generated_solution(x["generated_solution"])

    # 防止空推理让模型学到 <think>\n\n</think>（摆烂行为）
    if len(thoughts) < 5:
        thoughts = "I will solve the problem step by step."

    assistant = (
        f"{THINK_OPEN}\n{thoughts}\n{THINK_CLOSE}\n"
        f"{ANSWER_OPEN}\boxed{{{expected_answer}}}{ANSWER_CLOSE}"
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": problem},
        {"role": "assistant", "content": assistant},
    ]

print("✅ 格式契约函数就绪：system_prompt / chat_template / format_dataset")

✅ 格式契约函数就绪：system_prompt / chat_template / format_dataset


In [3]:
# ---- 本机 CPU 演示：格式契约长什么样（Colab/Kaggle 上可跳过本 cell）----
from transformers import AutoTokenizer

demo_tok = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B")
demo_tok.chat_template = chat_template

sample = format_dataset({
    "expected_answer": 42,
    "problem": "If a train travels 60 miles per hour for 2 hours, how far does it go?",
    "generated_solution": "<think>The train moves at 60 mph. In 2 hours it covers 60 × 2.</think>",
})
print(demo_tok.apply_chat_template(sample, tokenize=False, add_generation_prompt=True))
print("\n--- 推理太短的样本（自动补兜底话）---")
short = format_dataset({
    "expected_answer": 7,
    "problem": "What is 2+5?",
    "generated_solution": "<think>.</think>",
})
print(short[2]["content"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.
You will be provided with a problem. Please follow this format exactly (do not add any extra text outside the tags):
<think>
... step-by-step reasoning ...
</think>
<answer>\boxed{final numeric answer}</answer>
The <think> section can be short for easy problems, but must not be empty.
The final answer inside \boxed{} should be a single number.
<|eot_id|><|start_header_id|>user<|end_header_id|>

If a train travels 60 miles per hour for 2 hours, how far does it go?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<think>
The train moves at 60 mph. In 2 hours it covers 60 × 2.
</think>
<answer>oxed{42}</answer><|eot_id|><|start_header_id|>assistant<|end_header_id|>



--- 推理太短的样本（自动补兜底话）---
<think>
I will solve the problem step by step.
</think>
<answer>oxed{7}</answer>


## 5️⃣ 冷启动 SFT ②：数据 + 训练

数据侧两个过滤（原 notebook）：
1. **只留纯数字答案**：`pd.to_numeric` 判空——GRPO 阶段答案校验奖励只对数字有效；
2. **长度过滤**：`N ≤ max_seq_length × 0.8`——给后续 GRPO 的 `max_completion_length` 留余量。

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.model_name,
    max_seq_length = cfg.max_seq_length,
    load_in_4bit = False,   # False for LoRA 16bit
    fast_inference = True,  # Enable vLLM fast inference
    max_lora_rank = cfg.lora_rank,
    gpu_memory_utilization = 0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = cfg.lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = cfg.lora_rank * 2,      # *2 speeds up training（原注释）
    use_gradient_checkpointing = "unsloth",  # Reduces memory usage
    random_state = cfg.seed,
)
tokenizer.chat_template = chat_template

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset(cfg.cold_start_dataset, split=cfg.cold_start_split)
dataset = dataset.to_pandas()[["expected_answer", "problem", "generated_solution"]]

# 只保留 expected_answer 是数字的行（奖励可校验的前提）
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors="coerce").notnull()
dataset = dataset.iloc[np.where(is_number)[0]]

dataset["Messages"] = dataset.apply(format_dataset, axis=1)
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
dataset = dataset.loc[dataset["N"] <= cfg.max_seq_length * 0.8].copy()

from datasets import Dataset
dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize=False)
dataset = Dataset.from_pandas(dataset)
print(f"冷启动数据就绪：{len(dataset)} 条")

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = cfg.sft_batch,
        gradient_accumulation_steps = cfg.sft_accum,  # GA 模拟大 batch
        warmup_ratio = 0.04,
        num_train_epochs = cfg.sft_epochs,
        learning_rate = cfg.sft_lr,       # 长训降 2e-5（原注释）
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = cfg.seed,
        report_to = "none",
    ),
)
trainer.train()

## 6️⃣ GRPO 数据预处理

换 DAPO-Math-17k：prompt 包上 system_prompt，标准答案用 `extract_gt_answer` 归一化
（`\boxed{}` 优先 → 分数 → 最后一个数字），最后按 **90% 分位数**截断长度（原 notebook 做法）。

In [ ]:
from datasets import load_dataset
import numpy as np

# =========================================================
# ✅  RL Dataset Answer Extraction (for accuracy reward)
# =========================================================
_NUM_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_SLASH_FRAC_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?/[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_LATEX_FRAC_RE = re.compile(r"[-+]?\\(?:d?frac)\{[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?\}\{[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?\}")

def _extract_all_boxed(text: str):
    """提取所有 \boxed{...}（支持嵌套花括号）。"""
    if text is None:
        return []
    s = str(text); out = []; i = 0
    needle = r"\boxed{"
    while True:
        start = s.find(needle, i)
        if start == -1:
            break
        j = start + len(needle); depth = 1; buf = []
        while j < len(s) and depth > 0:
            ch = s[j]
            if ch == "{":
                depth += 1; buf.append(ch)
            elif ch == "}":
                depth -= 1
                if depth > 0: buf.append(ch)
            else:
                buf.append(ch)
            j += 1
        if depth == 0:
            out.append("".join(buf).strip()); i = j
        else:
            break
    return out

def extract_gt_answer(solution: str):
    """把各种答案格式归一化为可比较字符串：最后 \boxed{} 优先 → 分数 → 最后数字。"""
    if solution is None:
        return None
    s = str(solution)
    boxes = _extract_all_boxed(s)
    if boxes:
        return boxes[-1]
    m = None
    for m in _LATEX_FRAC_RE.finditer(s): pass
    if m: return m.group(0)
    for m in _SLASH_FRAC_RE.finditer(s): pass
    if m: return m.group(0)
    nums = _NUM_RE.findall(s.replace(",", ""))
    if nums: return nums[-1]
    return s.strip()

dataset = load_dataset(cfg.grpo_dataset, cfg.grpo_config, split="train")
dataset = dataset.map(lambda x: {
    "prompt": [{"role": "system", "content": system_prompt},
               {"role": "user",   "content": x["prompt"]}],
    "answer": extract_gt_answer(x["solution"]),
})

tokenized = dataset.map(
    lambda x: {"tokens": tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
    batched=True,
)
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})
maximum_length = int(np.quantile(tokenized["L"], 0.9))   # 90% 分位数截断
print("Max Length =", maximum_length)
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized
print(f"GRPO 数据就绪：{len(dataset)} 条")
print("\n--- 示例 prompt / 标准答案 ---")
print(dataset[0]["prompt"][1]["content"][:150], "\nanswer:", dataset[0]["answer"])

## 7️⃣ 奖励函数四件套

GRPO 与 PPO 的区别：**没有价值网络**，用"同组 N 条回答的奖励均值"作基线算优势。
奖励矩阵（原 notebook）：

| 奖励函数 | 信号 | 权重性质 |
|---|---|---|
| `basic_structure_reward` | 格式塑造：标签齐全/顺序/think 非空/答案只含 boxed/结尾干净 | 轻、稳定（空输出 -2 最差，非空有地板，防"摆烂不答"） |
| `check_answer_last_boxed` | **主信号**：`<answer>` 内 boxed 数值与标准答案比对，弱回退到全文数字 | 正确 +2，错误有负分 |
| `ngram_repetition_penalty` | 弱重复惩罚：3/4-gram 重复率 | 上限 -0.15，不喧宾夺主 |
| `print_samples_reward` | 采样监控：打印每组第 1 条，打分恒 0 | 纯观察 |

> ⚠️ 原 notebook 的关键经验：**格式惩罚太重会导致模型输出空串**（空串罚得比坏格式少），
> 所以空输出必须最差、坏格式设地板。

In [ ]:
import math
import re as _re

# -------------------------
# 0) 采样监控
# -------------------------
def print_samples_reward(prompts, completions, **kwargs):
    content = completions[0][0]["content"]
    print("\n" + "="*20 + " 🔍 [Monitor] Sample generated by model " + "="*20)
    print(f"❓ Prompt tail: ...{str(prompts[0])[-300:]}")
    print("-" * 10)
    print(f"🧠 Raw completion:\n{content[:1500]}")
    print("="*60 + "\n")
    return [0.0] * len(completions)

# -------------------------
# 1) 通用工具
# -------------------------
ALLOWED_TRAILING_STRS = [x for x in [
    getattr(tokenizer, "eos_token", None),
    getattr(tokenizer, "pad_token", None),
    "<|eot_id|>", "<|end_of_text|>"] if x]

def _strip_allowed_trailing(text: str) -> str:
    if text is None: return ""
    s = str(text).rstrip()
    changed = True
    while changed:
        changed = False
        for tok in ALLOWED_TRAILING_STRS:
            if s.endswith(tok):
                s = s[:-len(tok)].rstrip(); changed = True
    return s

def _count(hay, needle):
    return 0 if hay is None else str(hay).count(needle)

def _simple_tokens(s: str):
    return _re.findall(r"\w+|[^\w\s]", "" if s is None else str(s))

def _repetition_rate(toks, n):
    if len(toks) < n: return 0.0
    seen, rep = {}, 0
    for i in range(len(toks) - n + 1):
        g = tuple(toks[i:i+n])
        if g in seen: rep += 1
        else: seen[g] = 1
    return rep / max(1, len(toks) - n + 1)

def _normalize(s: str) -> str:
    if s is None: return ""
    return _re.sub(r"\s+", "", str(s).strip().strip("$").strip())

def _to_float(x):
    if x is None: return None
    s = _normalize(x).replace(",", "")
    if not s: return None
    if _re.fullmatch(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", s):
        try: return float(s)
        except: return None
    m = _re.fullmatch(r"([-+]?\d+(?:\.\d+)?)/([-+]?\d+(?:\.\d+)?)", s)
    if m:
        try: return float(m.group(1)) / float(m.group(2))
        except: pass
    nums = _NUM_RE.findall(s)
    if nums:
        try: return float(nums[-1])
        except: return None
    return None

def _similar(a, b):
    af, bf = _to_float(a), _to_float(b)
    if af is not None and bf is not None:
        return math.isclose(af, bf, rel_tol=1e-6, abs_tol=1e-9)
    return _normalize(a) == _normalize(b)

def _extract_last_boxed(text):
    boxes = _extract_all_boxed(text)
    return boxes[-1] if boxes else None

# -------------------------
# 2) 严格格式解析
# -------------------------
_STRICT_RE = _re.compile(
    r"^\s*<think>\s*(?P<think>.*?)\s*</think>\s*<answer>\s*(?P<answer>.*?)\s*</answer>\s*$",
    flags=_re.DOTALL)
_ONLY_BOXED_RE = _re.compile(r"^\s*\\boxed\{.*\}\s*$", flags=_re.DOTALL)

def _format_info(resp):
    raw = "" if resp is None else str(resp)
    stripped = _strip_allowed_trailing(raw)
    c_to, c_tc = _count(stripped, THINK_OPEN), _count(stripped, THINK_CLOSE)
    c_ao, c_ac = _count(stripped, ANSWER_OPEN), _count(stripped, ANSWER_CLOSE)
    p_to, p_tc = stripped.find(THINK_OPEN), stripped.find(THINK_CLOSE)
    p_ao, p_ac = stripped.find(ANSWER_OPEN), stripped.find(ANSWER_CLOSE)
    order_ok = (-1 not in (p_to, p_tc, p_ao, p_ac)) and (p_to < p_tc < p_ao < p_ac)
    m = _STRICT_RE.match(stripped)
    think_text = m.group("think") if m else ""
    answer_text = m.group("answer") if m else ""
    boxed_in_answer = _extract_last_boxed(answer_text) if answer_text else None
    only_boxed = bool(answer_text) and _ONLY_BOXED_RE.match(answer_text) is not None \
        and answer_text.count(r"\boxed{") == 1
    tail_after_answer = ""
    if p_ac != -1:
        tail_after_answer = stripped[p_ac + len(ANSWER_CLOSE):].strip()
    strict_ok = (m is not None and order_ok and (c_to == c_tc == c_ao == c_ac == 1)
                 and only_boxed and len(think_text.strip()) >= 5 and tail_after_answer == "")
    return {"stripped": stripped, "counts": (c_to, c_tc, c_ao, c_ac),
            "order_ok": order_ok, "strict_ok": strict_ok,
            "think_text": think_text, "answer_text": answer_text,
            "boxed_in_answer": boxed_in_answer, "only_boxed": only_boxed,
            "tail_after_answer": tail_after_answer}

# -------------------------
# 3) 奖励 A：结构塑造（空输出最差 + 坏格式地板，防摆烂）
# -------------------------
FMT_EMPTY, FMT_NONEMPTY_FLOOR = -2.0, -1.8
DUP_PENALTY_PER_KIND, TAIL_PENALTY = 0.9, 0.4

def basic_structure_reward(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        info = _format_info(c[0]["content"])
        s = info["stripped"]
        if len(s.strip()) == 0:
            scores.append(FMT_EMPTY); continue
        c_to, c_tc, c_ao, c_ac = info["counts"]
        score = 0.05                                    # 非空地板
        score += 0.15 if c_to >= 1 else -0.05           # 标签出现
        score += 0.15 if c_tc >= 1 else -0.05
        score += 0.15 if c_ao >= 1 else -0.05
        score += 0.15 if c_ac >= 1 else -0.05
        score += 0.10 if info["order_ok"] else -0.10    # 顺序
        score += 0.10 if len(info["think_text"].strip()) >= 5 else -0.05
        score += 0.10 if info["boxed_in_answer"] is not None else -0.05
        score += 0.05 if info["only_boxed"] else -0.05
        score += -TAIL_PENALTY if info["tail_after_answer"] else 0.05
        dup_kinds = sum(1 for x in (c_to, c_tc, c_ao, c_ac) if x > 1)
        score -= DUP_PENALTY_PER_KIND * dup_kinds
        scores.append(max(FMT_NONEMPTY_FLOOR, min(score, 1.0)))
    return scores

# -------------------------
# 4) 奖励 B：答案校验（主信号）
# -------------------------
def _broadcast_answers(answer, n):
    if isinstance(answer, str):
        return [answer] * n
    if isinstance(answer, (list, tuple)):
        return list(answer) if len(answer) == n else [str(answer)] * n
    return [str(answer)] * n

def check_answer_last_boxed(prompts, completions, answer, **kwargs):
    scores = []
    answer_list = _broadcast_answers(answer, len(completions))
    for c, true_ans in zip(completions, answer_list):
        info = _format_info(c[0]["content"])
        pred = info["boxed_in_answer"]                   # ① 最高优先：<answer> 内最后 boxed
        if pred is None and info["answer_text"]:         # ② 次之：answer_text 里的数字/分数
            cand = None; m = None
            for m in _LATEX_FRAC_RE.finditer(info["answer_text"]): pass
            if m: cand = m.group(0)
            if cand is None:
                for m in _SLASH_FRAC_RE.finditer(info["answer_text"]): pass
                if m: cand = m.group(0)
            if cand is None:
                nums = _NUM_RE.findall(info["answer_text"].replace(",", ""))
                if nums: cand = nums[-1]
            pred = cand
        if pred is None:                                 # ③ 弱回退：全文最后 boxed / 最后数字
            pred = _extract_last_boxed(info["stripped"])
        if pred is None:
            nums = _NUM_RE.findall(info["stripped"].replace(",", ""))
            pred = nums[-1] if nums else None
        if pred is None:
            scores.append(-0.5); continue                 # 什么都没答
        scores.append(2.0 if _similar(pred, true_ans) else -1.0)
    return scores

# -------------------------
# 5) 奖励 C：弱重复惩罚
# -------------------------
def ngram_repetition_penalty(prompts, completions, **kwargs):
    scores = []
    COEF_3, COEF_4, CAP = 0.005, 0.008, -0.15
    for c in completions:
        toks = _simple_tokens(c[0]["content"])
        penalty = -(COEF_3 * _repetition_rate(toks, 3) + COEF_4 * _repetition_rate(toks, 4))
        scores.append(max(penalty, CAP))
    return scores

# -------------------------
# 6) 预留：think/答案一致性（原 notebook 注释为可省略）
# -------------------------
def think_boxed_alignment_reward(prompts, completions, **kwargs):
    return [0.0] * len(completions)

print("✅ 奖励函数就绪：structure / answer-boxed / repetition / monitor（+1 预留）")

## 8️⃣ GRPO 训练

GRPOConfig 与原 notebook 一致：
- `optim="paged_adamw_8bit"`——RL 阶段显存更紧，分页优化器把状态换页到 CPU；
- `num_generations=6`——每步采样 6 条回答组成比较组（**OOM 先降它**）；
- `max_prompt_length + max_completion_length = max_seq_length`；
- lr 5e-6 + cosine + warmup 0.1——RL 学习率比 SFT 低一个数量级。

In [ ]:
from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
max_prompt_length = maximum_length + 1          # + 1 just in case!
max_completion_length = cfg.max_seq_length - max_prompt_length

training_args = GRPOConfig(
    temperature = cfg.temperature,
    learning_rate = cfg.grpo_lr,                 # RL：比 SFT 低一个数量级
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = cfg.grpo_batch,
    gradient_accumulation_steps = cfg.grpo_accum,  # Increase to 4 for smoother training
    num_generations = cfg.num_generations,         # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = cfg.max_steps,
    save_steps = 20,
    save_total_limit = 1,
    save_strategy = "steps",
    report_to = "wandb",
    output_dir = "./grpo_output",
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        print_samples_reward,
        basic_structure_reward,        # 奖励 1：结构塑造（轻、稳定）
        check_answer_last_boxed,       # 奖励 2：主信号（boxed 数值校验 + 弱回退）
        ngram_repetition_penalty,      # 奖励 4：弱重复惩罚
        # think_boxed_alignment_reward, # 奖励 3：占位（原 notebook 可省略）
    ],
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

## 9️⃣ 训练观察与推理自测

观察三件事：`[Monitor]` 打印的采样格式是否逐渐收敛、奖励曲线是否上升、答案正确率。
自测用 `fast_generate`（Unsloth vLLM 后端，比 HF generate 快一个量级）。

In [ ]:
text = "What is the sqrt of 101?"

sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [tokenizer.apply_chat_template([{"role": "user", "content": text}],
                                   tokenize=False, add_generation_prompt=True)],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

print(output)

## 🔟 保存与 GGUF 导出（Qwopus 风格收尾）

GRPO 后同样走 LoRA → merged 16bit → GGUF 三件套（原 notebook 用 `trainer.model.save_lora`）。

In [ ]:
# ==========================================
# ✅ 1. Save LoRA Adapter After Training
# ==========================================
print("💾 Saving GRPO after LoRA...")
trainer.model.save_lora("grpo_saved_lora_final")
print("✅ Saving completed!")

# ---- 2. 合并 16bit 并推送（可选，需 HF_TOKEN）----
# model.push_to_hub_merged("你的用户名/Llama-3.2-3B-R1-GRPO",
#                          tokenizer, save_method="merged_16bit", token=HF_TOKEN)

# ---- 3. GGUF 三档量化并推送 ----
# model.push_to_hub_gguf("你的用户名/Llama-3.2-3B-R1-GRPO-GGUF",
#                        tokenizer, quantization_method=["q4_k_m", "q8_0", "bf16"],
#                        token=HF_TOKEN)

## 1️⃣1️⃣ 附录

### A. 故障排查

| 症状 | 可能原因 | 处理 |
|---|---|---|
| GRPO OOM | vLLM 引擎 + 训练模型同卡 | 降 `num_generations`；确认 `UNSLOTH_VLLM_STANDBY=1` 已设 |
| 模型输出空串 / 只答格式标签 | 格式惩罚过重 | 本教程已按原 notebook 经验设空输出最差 + 坏格式地板 |
| 格式收敛但答案不对 | 主信号弱于格式信号 | 调大 `check_answer_last_boxed` 权重或降结构奖励 |
| 冷启动后格式仍乱 | SFT epoch 不够 / 数据格式过滤太松 | 检查 `N ≤ 0.8×seq` 过滤与 `format_dataset` 产物 |

### B. 与中篇/上篇的关系

- 冷启动 SFT 的 `adamw_8bit + 梯度累积` 与上篇省钱五件套完全一致；
- 全篇代码来源：[train_code/Llama-3.2-3B-R1-Zero-GRPO.ipynb](../../Jackrong-llm-finetuning-guide/train_code/Llama-3.2-3B-R1-Zero-GRPO.ipynb)（原 notebook，奖励函数为完整 13.7K 版，本教程为忠实精简版）；
- RL 数据 `open-r1/DAPO-Math-17k-Processed` 与冷启动数据 `unsloth/OpenMathReasoning-mini` 的详情见 `food/post-training.md` 第八节。

### C. 参考

1. DeepSeek-R1-Zero / GRPO（Shao et al., 2024）—— 群组相对优势的思想源头
2. jackrong《Jackrong-llm-finetuning-guide》—— 本教程代码来源
3. `food/post-training.md` —— 数据集清单

---

🎉 **中篇结束**。你已掌握"冷启动 SFT 教格式 → GRPO 多奖励函数教推理"的完整两阶段。上篇（省钱 SFT）见 `cook_qwopus_sft.ipynb`，下篇（缝合+愈合）见 `cook_merge_heal.ipynb`。